# Step 7-2: 측면(Aspect) 매칭

**방법 C — 하이브리드 ABSA의 두 번째 단계**

각 문장이 어떤 측면(사이즈, 핏, 재질, 색상 등)을 다루는지 매칭.
한 문장이 여러 측면에 매칭될 수 있음 (예: "사이즈 좋고 색감도 예뻐요" → size, color 둘 다).

## 입력 / 출력
| 파일 | 내용 |
|------|------|
| `step7_1_sentences.parquet` | 7-1 문장 분할 결과 (~159만 문장) |
| → `aspect_dict.json` | 측면 사전 (재사용용) |
| → `step7_2_aspect_pairs.parquet` | (문장, 측면) 페어 — 7-3 감성 분류 입력 |
| → `step7_2_unmatched.parquet` | 매칭 실패 문장 (분석/보강용) |

## 매칭 전략
1. **측면 사전 정의** — 카테고리별 어근 키워드
2. **Kiwi 형태소 분석** — step6과 동일 (보호어 등록 → NNG/VA/XR 추출)
3. **토큰 단위 정확 매칭** — `tok.form ∈ ASPECT_DICT[cat]`이면 매칭
4. **다중 매칭 허용** — 한 문장이 여러 측면에 매칭됨

## 7-3에서의 활용
한 행 = 한 (문장, 측면) 페어 → 그 페어에 대해 감성 분류 → 측면별 긍/부 비율 집계


In [4]:
!pip install -q kiwipiepy

In [5]:
from google.colab import drive

import os
import json
import time
import numpy as np
import pandas as pd
from collections import Counter
from kiwipiepy import Kiwi
from tqdm.auto import tqdm

In [6]:
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/sparta/tp4/review_analysis/data/'
INPUT_PATH = OUTPUT_DIR + 'step7_1_sentences.parquet'

print(f'INPUT     : {INPUT_PATH}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

Mounted at /content/drive
INPUT     : /content/drive/MyDrive/sparta/tp4/review_analysis/data/step7_1_sentences.parquet
OUTPUT_DIR: /content/drive/MyDrive/sparta/tp4/review_analysis/data/


## 7-2-1. 데이터 로드

In [7]:
df_sents = pd.read_parquet(INPUT_PATH)
df_sents['sentence'] = df_sents['sentence'].fillna('').astype(str)
df_sents = df_sents[df_sents['sentence'].str.len() >= 2].reset_index(drop=True)

print(f'총 문장 수: {len(df_sents):,}')
print(f'고유 리뷰 수: {df_sents["리뷰번호"].nunique():,}')
print(f'\n[컬럼]')
print(df_sents.columns.tolist())
print(f'\n[topic_id 분포]')
print(df_sents['topic_id'].value_counts().head(5))
n_outlier = (df_sents['topic_id'] == -1).sum()
print(f'... outlier(-1): {n_outlier:,}건 ({n_outlier/len(df_sents)*100:.1f}%)')

총 문장 수: 1,531,645
고유 리뷰 수: 598,301

[컬럼]
['리뷰번호', 'sent_id', 'sentence', 'n_sents', 'topic_id', 'sent_len']

[topic_id 분포]
topic_id
-1     503147
 6      83767
 2      75516
 46     54637
 15     52726
Name: count, dtype: int64
... outlier(-1): 503,147건 (32.9%)


## 7-2-2. 측면 사전 (ASPECT_DICT) 정의

PROTECTED_WORDS는 step6 토픽 분석용이고, ABSA에는 추가 카테고리(색상·가격·배송 등)가 필요.
ASPECT_DICT를 별도로 정의하되, **어근 형태**로 작성 (Kiwi 토큰화 결과와 매칭).

### 어근 작성 원칙
| 원형 | 어근 (ASPECT_DICT 등록) | Kiwi 토큰 결과 |
|---|---|---|
| 두께가 / 두께를 | `두께` | `두께` (NNG) |
| 따뜻해요 | `따뜻` | `따뜻` (XR) |
| 두꺼워요 | `두껍` | `두껍` (VA-I) |
| 편해요 | `편하` | `편하` (VA) |

### 카테고리 9개 (확장 가능)
- `fit` — 핏/실루엣
- `size` — 사이즈
- `fabric` — 원단/재질
- `color` — 색상
- `detail` — 디자인 디테일
- `season` — 계절/온도
- `comfort` — 착용감
- `price` — 가격
- `delivery` — 배송/포장
- `durability` — 내구성/세탁

> 한 키워드가 여러 카테고리에 들어갈 수 있음 (예: `따뜻`은 `fabric`이자 `season`).
> 우리는 **다중 매칭 허용** — 한 문장이 여러 측면에 매칭되면 모두 기록.

In [8]:
# ASPECT_DICT — 측면별 어근 키워드
# 추후 매칭률 진단 후 보강 (7-2-6 결과 보고 추가/삭제)
ASPECT_DICT = {
    # 1. 핏/실루엣
    'fit': {
        '오버핏', '오버사이즈', '루즈핏', '레귤러핏', '슬림핏', '와이드핏',
        '세미오버', '박시핏', '머슬핏', '크롭핏', '박시', '핏',
        '벌룬핏', '부츠컷', '스탠다드핏', '세미와이드',
        '핏감', # +
    },
    # 2. 사이즈
    'size': {
        '사이즈', '치수', '크기', '길이', '품',
        '정사이즈', '큼', '작', '크', '맞',
        '어깨', '허리', '허벅지', '엉덩이',
        '짧', '넉넉', '타이트', # +
    },
    # 3. 원단/재질
    'fabric': {
        '재질', '원단', '소재', '촉감',
        '두께', '두께감', '두툼', '도톰', '두껍', '얇',
        '기모', '면', '폴리', '코듀로이', '니트', '데님', '청',
        '스판', '신축성', '쫀쫀', '탄탄',
        '보풀', '거치',
        '비침', '비치', # +
        '반딱', '반짝', '광택','윤기', # +
        '매끄럽', '부드럽', '딱딱', '빳빳', '까실', '사각' # +
        '가죽', # +
    },
    # 4. 색상
    'color': {
        '색감', '색상', '컬러',
        '검정', '블랙', '화이트', '네이비', '베이지', '카키',
        '오트밀', '그레이', '브라운', '아이보리',
        '진하', '연하', '흐리', '탁하',
        '화사', '산뜻', '오묘', '은은', '선명',
        '톤', # +
        '핑크', '초록', '빨강', '파랑', '노랑', '보라', '민트',  # +v2
        '레드', '블루', '그린', '옐로우', '오렌지',              # +v2
        '와인', '연두', '하늘', '버건디', # +v2

    },
    # 5. 디자인 디테일
    'detail': {
        '디자인', '스타일',
        '봉제', '스티치', '주머니', '포켓', '밑단', '소매',
        '배색', '단추', '지퍼', '자수', '프린팅', '밴딩',
        '시보리', '워싱', '레이어드',
        '카라', '넥라인', '목폴라', '반폴라',
        '예쁘', '이쁘', '멋지', '깔끔', '단정',
        '무지', '밋밋', # +
        '카고', '패턴', '체크', '스트라이프', '사선', # +
        '로고', '그래픽', # +
        '트임', '절개', '핀턱', # +
    },
    # 6. 계절/온도
    'season': {
        '여름', '겨울', '봄', '가을', '간절기', '환절기',
        '쌀쌀', '시원', '따뜻', '따듯', '포근', '보온',
        '날씨', '봄가을', '초겨울', '봄철', '초봄',
        '덥', '춥',
        '사철', '사계절', # +
    },
    # 7. 착용감
    'comfort': {
        '편하', '편안', '편함', '불편',
        '가볍', '무겁', '거추',
        '답답', '갑갑', '텁텁', '쾌적', # +
    },
    # 8. 가격
    'price': {
        '가격', '가성비', '비싸', '비쌈', '싸', '저렴', '비용', '값',
    },
    # 9. 배송/포장
    'delivery': {
        '배송', '포장', '도착', '배달',
        '빠르', '늦', # +
    },
    # 10. 내구성/세탁
    'durability': {
        '세탁', '변형', '늘어나', '늘어남', '튼튼', '오래',
        '묻', '이염', '변색', # +
        '헤짐', '닳' # +
    },
}

# 카테고리/키워드 통계
n_cat = len(ASPECT_DICT)
n_kw  = sum(len(v) for v in ASPECT_DICT.values())
print(f'카테고리 수: {n_cat}')
print(f'전체 키워드 수: {n_kw}')
print(f'\n[카테고리별 키워드 수]')
for cat, kws in ASPECT_DICT.items():
    print(f'  {cat:<12} {len(kws):>3}개')

# JSON으로 저장 (재사용용)
ASPECT_PATH = OUTPUT_DIR + 'aspect_dict.json'
with open(ASPECT_PATH, 'w', encoding='utf-8') as f:
    json.dump({k: sorted(v) for k, v in ASPECT_DICT.items()}, f,
              ensure_ascii=False, indent=2)
print(f'\n저장: {ASPECT_PATH}')

카테고리 수: 10
전체 키워드 수: 203

[카테고리별 키워드 수]
  fit           17개
  size          17개
  fabric        35개
  color         39개
  detail        38개
  season        21개
  comfort       11개
  price          8개
  delivery       6개
  durability    11개

저장: /content/drive/MyDrive/sparta/tp4/review_analysis/data/aspect_dict.json


In [9]:
# 키워드 → 카테고리 역매핑 (한 키워드가 여러 카테고리에 속할 수 있음)
KEYWORD_TO_ASPECTS = {}
for cat, kws in ASPECT_DICT.items():
    for kw in kws:
        KEYWORD_TO_ASPECTS.setdefault(kw, []).append(cat)

# 다중 카테고리 키워드 확인
multi_cat_kws = {kw: cats for kw, cats in KEYWORD_TO_ASPECTS.items() if len(cats) > 1}
print(f'다중 카테고리 키워드: {len(multi_cat_kws)}개')
for kw, cats in list(multi_cat_kws.items())[:10]:
    print(f'  {kw!r:<10} → {cats}')

다중 카테고리 키워드: 0개


## 7-2-3. Kiwi 토크나이저 + 보호어 등록

step6/7-1과 **동일한 보호어** 등록 → 토큰화 일관성 보장.

> 보호어 변경 시 step6, 7-1, 7-2 모두 동기화 필요.
> 권장: `protected_words.json`으로 분리 (이건 별도로 정리 가능).

In [10]:
with open('/content/drive/MyDrive/sparta/tp4/review_analysis/data/protected_words.json', 'r', encoding='utf-8') as f:
    PROTECTED_WORDS = json.load(f)

all_protected = sum(PROTECTED_WORDS.values(), [])
print(f'전체 보호어: {len(all_protected)}개')

kiwi = Kiwi()
for word in all_protected:
    kiwi.add_user_word(word, 'NNG', score=5)
print(f'Kiwi 사용자 사전 등록 완료')

전체 보호어: 104개
Kiwi 사용자 사전 등록 완료


## 7-2-4. 매칭 함수 정의 + 테스트

### 핵심 로직
1. 문장 → Kiwi 토큰화
2. 명사/형용사/어근(`NNG`/`NNP`/`VA`/`XR`/`SL`)만 추출
3. 각 토큰이 `KEYWORD_TO_ASPECTS`에 있는지 확인
4. 매칭된 (카테고리, 키워드) 페어를 `set`으로 모음 → 중복 제거

In [11]:
ALLOWED_POS = {'NNG', 'NNP', 'VA', 'XR', 'SL', 'VV'}


def match_aspects(text: str) -> list:
    """문장에서 측면 키워드를 매칭하여 [(category, keyword), ...] 리스트 반환.

    한 문장이 여러 측면에 매칭될 수 있음 (다중 매칭 허용).
    """
    if not text or not isinstance(text, str):
        return []

    try:
        tokens = kiwi.tokenize(text)
    except Exception:
        return []

    matched = set()
    for tok in tokens:
        # prefix 기반 — VA-I, VV-I, NNG, NNP 등 모든 변종 포함
        if not (tok.tag.startswith(('NN', 'VA', 'VV', 'XR')) or tok.tag == 'SL'):
            continue
        cats = KEYWORD_TO_ASPECTS.get(tok.form)
        if cats:
            for cat in cats:
                matched.add((cat, tok.form))

    return sorted(matched)


# ─────────────────────────────────────────────
# 매칭 테스트
# ─────────────────────────────────────────────
test_sents = [
    '사이즈 좋고 색감도 예뻐요',
    '기모가 두꺼운데 따뜻하고 가벼워요',
    '오버핏이라 좋아요',
    '배송 빨라요',
    '가성비 최고',
    '세탁하니까 보풀이 생기네요',
    '예쁜데 비싸요',
    '잘 받았어요',                    # 매칭 0개 예상
    '검정색이 정말 진해서 좋네요',
    '여름에 시원하게 입어요',
]

print('[매칭 테스트]')
for text in test_sents:
    aspects = match_aspects(text)
    print(f'\n  문장: {text}')
    if aspects:
        for cat, kw in aspects:
            print(f'    → ({cat}, {kw})')
    else:
        print(f'    → (매칭 없음)')

[매칭 테스트]

  문장: 사이즈 좋고 색감도 예뻐요
    → (color, 색감)
    → (detail, 예쁘)
    → (size, 사이즈)

  문장: 기모가 두꺼운데 따뜻하고 가벼워요
    → (comfort, 가볍)
    → (fabric, 기모)
    → (fabric, 두껍)
    → (season, 따뜻)

  문장: 오버핏이라 좋아요
    → (fit, 오버핏)

  문장: 배송 빨라요
    → (delivery, 배송)
    → (delivery, 빠르)

  문장: 가성비 최고
    → (price, 가성비)

  문장: 세탁하니까 보풀이 생기네요
    → (durability, 세탁)
    → (fabric, 보풀)

  문장: 예쁜데 비싸요
    → (detail, 예쁘)
    → (price, 비싸)

  문장: 잘 받았어요
    → (매칭 없음)

  문장: 검정색이 정말 진해서 좋네요
    → (color, 검정)
    → (color, 진하)

  문장: 여름에 시원하게 입어요
    → (season, 시원)
    → (season, 여름)


## 7-2-5. 전체 문장에 매칭 적용

159만 문장 × Kiwi 토큰화 → ~25-35분 예상.

In [12]:
print('측면 매칭 중...')
t0 = time.time()

tqdm.pandas()
df_sents['matched_aspects'] = df_sents['sentence'].progress_apply(match_aspects)
df_sents['n_aspects'] = df_sents['matched_aspects'].apply(len)

elapsed = (time.time() - t0) / 60
print(f'\n매칭 완료: {elapsed:.1f}분')

# 매칭 통계
n_matched = (df_sents['n_aspects'] >= 1).sum()
n_unmatched = (df_sents['n_aspects'] == 0).sum()
total = len(df_sents)

print(f'\n[매칭 결과]')
print(f'  매칭 성공: {n_matched:,}건 ({n_matched/total*100:.1f}%)')
print(f'  매칭 실패: {n_unmatched:,}건 ({n_unmatched/total*100:.1f}%)')

print(f'\n[문장당 측면 수 분포]')
print(df_sents['n_aspects'].value_counts().sort_index())

측면 매칭 중...


  0%|          | 0/1531645 [00:00<?, ?it/s]


매칭 완료: 14.9분

[매칭 결과]
  매칭 성공: 960,061건 (62.7%)
  매칭 실패: 571,584건 (37.3%)

[문장당 측면 수 분포]
n_aspects
0     571584
1     539101
2     277780
3     102787
4      30916
5       7383
6       1616
7        352
8         83
9         27
10        10
11         3
12         2
14         1
Name: count, dtype: int64


## 7-2-6. 매칭 진단

매칭 결과를 4가지 관점에서 점검:
1. **카테고리별 매칭 빈도** — 어느 측면이 가장 자주 언급되는지
2. **카테고리별 키워드 분포** — 각 측면에서 어떤 키워드가 주도하는지
3. **토픽 × 카테고리 정렬** — 토픽이 측면을 잘 대표하는지 (예: 사이즈 토픽 → size 매칭률 높아야)
4. **매칭 실패 문장 샘플** — 사전 보강 힌트

In [13]:
# (1) 카테고리별 매칭 빈도
cat_counter = Counter()
for aspects in df_sents['matched_aspects']:
    for cat, _ in aspects:
        cat_counter[cat] += 1

print('[카테고리별 매칭 문장 수 (중복 허용)]')
total_matched_pairs = sum(cat_counter.values())
for cat, cnt in cat_counter.most_common():
    pct = cnt / total * 100
    print(f'  {cat:<12} {cnt:>8,}건 ({pct:5.1f}%)')
print(f'  {"--" * 15}')
print(f'  {"전체 (문장,측면) 페어":<12} {total_matched_pairs:>8,}건')

[카테고리별 매칭 문장 수 (중복 허용)]
  detail        323,082건 ( 21.1%)
  size          320,608건 ( 20.9%)
  fabric        226,324건 ( 14.8%)
  fit           169,897건 ( 11.1%)
  season        143,904건 (  9.4%)
  color         123,617건 (  8.1%)
  price         104,214건 (  6.8%)
  comfort        86,390건 (  5.6%)
  delivery       55,871건 (  3.6%)
  durability     22,932건 (  1.5%)
  ------------------------------
  전체 (문장,측면) 페어 1,576,839건


In [14]:
# (2) 카테고리별 상위 키워드 (어떤 키워드가 매칭을 주도하는지)
cat_kw_counter = {cat: Counter() for cat in ASPECT_DICT}
for aspects in df_sents['matched_aspects']:
    for cat, kw in aspects:
        cat_kw_counter[cat][kw] += 1

print('[카테고리별 상위 키워드 Top 5]')
for cat in ASPECT_DICT.keys():
    top = cat_kw_counter[cat].most_common(5)
    if not top:
        print(f'\n  {cat}: (매칭 없음)')
        continue
    print(f'\n  {cat}:')
    for kw, cnt in top:
        print(f'    {kw!r:<12} {cnt:>7,}건')

[카테고리별 상위 키워드 Top 5]

  fit:
    '핏'          103,028건
    '오버핏'         48,963건
    '박시'           5,516건
    '오버사이즈'        3,375건
    '핏감'           2,843건

  size:
    '사이즈'        101,399건
    '크'           73,691건
    '맞'           37,306건
    '길이'          18,547건
    '허리'          15,776건

  fabric:
    '재질'          39,485건
    '두껍'          28,585건
    '기모'          28,213건
    '얇'           27,234건
    '두께'          15,765건

  color:
    '색감'          46,406건
    '색상'          26,781건
    '블랙'           6,346건
    '그레이'          5,755건
    '컬러'           5,081건

  detail:
    '이쁘'         123,303건
    '예쁘'          85,043건
    '디자인'         26,882건
    '레이어드'        14,118건
    '깔끔'          10,448건

  season:
    '여름'          31,592건
    '따뜻'          26,885건
    '겨울'          17,231건
    '가을'          12,404건
    '덥'           10,628건

  comfort:
    '편하'          65,463건
    '가볍'           9,933건
    '편안'           3,723건
    '불편'           3,489건
    '무겁'           2,53

In [15]:
# (3) 토픽 × 카테고리 매칭률 (토픽이 측면을 잘 대표하는지)
# 큰 토픽 상위 10개에 대해 각 카테고리 매칭 비율 확인
top_topics = (
    df_sents[df_sents['topic_id'] != -1]
    .groupby('topic_id').size()
    .sort_values(ascending=False).head(10).index.tolist()
)

print('[상위 10개 토픽의 카테고리별 매칭률]')
print(f'{"topic_id":<10}{"문장수":<10}', end='')
for cat in ASPECT_DICT.keys():
    print(f'{cat[:6]:<7}', end='')
print()

for tid in top_topics:
    mask = df_sents['topic_id'] == tid
    n = mask.sum()
    print(f'{tid:<10}{n:<10,}', end='')
    for cat in ASPECT_DICT.keys():
        cnt = sum(
            1 for aspects in df_sents.loc[mask, 'matched_aspects']
            if any(c == cat for c, _ in aspects)
        )
        pct = cnt / n * 100
        marker = '*' if pct >= 30 else ' '
        print(f'{pct:5.1f}{marker} ', end='')
    print()
print('\n* >= 30% — 토픽의 주요 측면')

[상위 10개 토픽의 카테고리별 매칭률]
topic_id  문장수       fit    size   fabric color  detail season comfor price  delive durabi 
6         83,767      9.8   15.8    8.5    6.2   20.6    6.0    8.0    5.1    0.5    0.9  
2         75,516      6.9   10.5    7.9    6.2   17.0    4.7    3.4    6.0   37.5*   0.7  
46        54,637     12.9   36.6*   5.7    3.4   12.7    2.4    2.2    2.2    0.4    0.5  
15        52,726     12.0   12.3   12.7    5.5   17.6    7.2    4.2    7.4    0.5    1.0  
13        47,453     12.0   12.5   45.0*   5.4   14.8   23.6    4.8    4.8    0.4    2.3  
63        43,187      6.8    9.3   11.1    5.4   15.6    1.4    2.8    5.5    1.9   15.9  
19        38,623      7.2    8.9    6.9   36.3*  18.9    2.7    3.7    3.3    0.5    0.7  
64        36,645     16.9   20.9   15.9   18.0   34.4*   0.3    1.7    1.5    0.1    0.4  
51        30,399     11.1    8.5    8.5    6.3   18.6    0.3    1.6   33.6*   0.4    0.3  
12        28,772     10.4   13.0   15.3    4.4   15.4   39.4*   8.1

In [16]:
# (4) 매칭 실패 문장 샘플 (사전 보강 힌트)
unmatched = df_sents[df_sents['n_aspects'] == 0]
print(f'매칭 실패: {len(unmatched):,}건 ({len(unmatched)/total*100:.1f}%)')

# 길이별 분포 — 짧은 문장은 어쩔 수 없지만, 긴 문장은 사전 부족 신호
print(f'\n[매칭 실패 문장 길이 분포]')
print(unmatched['sentence'].str.len().describe())

# 길이 5자 이상인 매칭 실패 샘플 (사전 보강에 활용)
unmatched_long = unmatched[unmatched['sentence'].str.len() >= 5]
print(f'\n[5자 이상 매칭 실패 샘플 30건]')
for s in unmatched_long['sentence'].sample(min(30, len(unmatched_long)),
                                            random_state=42).tolist():
    print(f'  - {s}')

매칭 실패: 571,584건 (37.3%)

[매칭 실패 문장 길이 분포]
count    571584.000000
mean         12.502283
std           7.638084
min           2.000000
25%           7.000000
50%          11.000000
75%          16.000000
max         153.000000
Name: sentence, dtype: float64

[5자 이상 매칭 실패 샘플 30건]
  - 흰 옷 위에 입으면 털이 좀 뭍는 게 아쉽지만
  - 28입는데
  - 기장이 살짝 길긴하네요
  - 감사합니다
  - 너무 잘 구매한거같아요
  - 색깔도 선호색이라 좋아요
  - 근데 빠니까 줄어서 재구매했어요
  - 막 입기 넘 좋습니다
  - 좋군요!!
  - 전반적으로 만족 합니당
  - 색도 맘에 들고 낙낙해서 좋아요
  - 아들 사줬능데 맘에 든다네여 ㅋㅋ
  - 엄청 오래 걸려서 받았는데
  - 딱 지갑 하나 들어가는 공간입니다.
  - 일본인 남친 따라 샀어요
  - 마음에 들어요!!
  - 너무 아쉽네요..
  - 만족합니다
  - 나쁘지 않습니다
  - 매우 만족스럽고 입었을 때 부담이 없었습니다.
  - 절에 들어가야될것만 같은ㅋㅋ 정말 벌룬벌룬합니다
  - 여자친구랑 막 입는 커플 바지로 삼
  - 주변에 마구 추천해주고 싶어용
  - 너무 좋았어요
  - 그냥그래요
  - 재구매 의사 있습니다
  - 동네 전투용으로 구매했습니다
  - 변화없고..
  - 감사합니다
  - 털이라서 먼지 조금 붙는것만 빼면 괜찮아요


## 7-2-7. (문장, 측면) 페어로 explode

7-3 감성 분류를 위해 **한 행 = 한 (문장, 측면) 페어** 형태로 펼친다.
한 문장이 3개 측면에 매칭되면 → 3행으로 분리.

In [17]:
# 매칭된 문장만 explode
df_matched = df_sents[df_sents['n_aspects'] >= 1].copy()

# 보존할 메타 컬럼
front_cols = ['리뷰번호', 'sent_id', 'sentence', 'n_sents', 'topic_id']
front_cols = [c for c in front_cols if c in df_matched.columns]
other_meta = [c for c in df_matched.columns
              if c not in front_cols + ['matched_aspects', 'n_aspects', 'sent_len']]

# explode
df_pairs = df_matched[front_cols + other_meta + ['matched_aspects']].explode('matched_aspects').reset_index(drop=True)

# 튜플 → 두 컬럼 분리
df_pairs[['aspect_category', 'aspect_keyword']] = pd.DataFrame(
    df_pairs['matched_aspects'].tolist(), index=df_pairs.index
)
df_pairs = df_pairs.drop(columns=['matched_aspects'])

# pair_id (한 문장 내 측면 순번)
df_pairs['pair_id'] = df_pairs.groupby(['리뷰번호', 'sent_id']).cumcount()

# 컬럼 순서 정리
front = ['리뷰번호', 'sent_id', 'pair_id', 'sentence',
         'aspect_category', 'aspect_keyword', 'topic_id']
front = [c for c in front if c in df_pairs.columns]
others = [c for c in df_pairs.columns if c not in front]
df_pairs = df_pairs[front + others]

print(f'페어 수: {len(df_pairs):,}')
print(f'  (매칭 성공 문장 {len(df_matched):,}건 × 평균 {len(df_pairs)/len(df_matched):.2f}측면)')
print(f'\n[샘플 10건]')
df_pairs.head(10)

페어 수: 1,576,839
  (매칭 성공 문장 960,061건 × 평균 1.64측면)

[샘플 10건]


,리뷰번호,sent_id,pair_id,sentence,aspect_category,aspect_keyword,topic_id,n_sents
0,51564285,1,0,편하고 그냥 입기에도 조아요,comfort,편하,16,2
1,46679669,1,0,진짜 많이 커요,size,크,16,3
2,46679669,2,0,조금은 두꺼운 감이 있어요,fabric,두껍,16,3
3,49013088,1,0,정말 크고 두꺼워요,fabric,두껍,16,2
4,49013088,1,1,정말 크고 두꺼워요,size,크,16,2
5,12731504,0,0,이뻐요.,detail,이쁘,43,4
6,33164355,2,0,"품질 ,가격 최곱니다",price,가격,-1,3
7,64332738,0,0,통도 너무 크고 길이가 너무 길어요ㅜㅜ,size,길이,-1,2
8,64332738,0,1,통도 너무 크고 길이가 너무 길어요ㅜㅜ,size,크,-1,2
9,64332738,1,0,그래도 너무 이뻐서 다시 살까 생각중입니다ㅜ,detail,이쁘,-1,2


## 7-2-8. 저장

세 파일을 저장:
- `step7_2_aspect_pairs.parquet` — (문장, 측면) 페어 (7-3 입력)
- `step7_2_unmatched.parquet` — 매칭 실패 문장 (사전 보강 시 참고)
- `aspect_dict.json` — 측면 사전 (7-3, 7-4에서 재사용)

In [18]:
# (1) 페어 저장
pairs_path = OUTPUT_DIR + 'step7_2_aspect_pairs.parquet'
df_pairs.to_parquet(pairs_path, index=False)
print(f'저장: {pairs_path}')
print(f'  총 {len(df_pairs):,}페어')
print(f'  메모리: {df_pairs.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

# (2) 매칭 실패 저장 (사전 보강용)
unmatched_path = OUTPUT_DIR + 'step7_2_unmatched.parquet'
unmatched_save = unmatched.drop(columns=['matched_aspects', 'n_aspects'], errors='ignore')
unmatched_save.to_parquet(unmatched_path, index=False)
print(f'\n저장: {unmatched_path}')
print(f'  총 {len(unmatched_save):,}문장')

# (3) 카테고리별 매칭 통계 (보고용)
stats_rows = []
for cat in ASPECT_DICT.keys():
    cnt = cat_counter.get(cat, 0)
    top_kws = cat_kw_counter[cat].most_common(3)
    stats_rows.append({
        'category': cat,
        'matched_sentences': cnt,
        'pct_of_total': round(cnt / total * 100, 2),
        'top_keywords': ', '.join(f'{kw}({c})' for kw, c in top_kws),
    })
stats_df = pd.DataFrame(stats_rows).sort_values('matched_sentences', ascending=False)
stats_path = OUTPUT_DIR + 'step7_2_match_stats.csv'
stats_df.to_csv(stats_path, index=False, encoding='utf-8-sig')
print(f'\n저장: {stats_path}')
stats_df

저장: /content/drive/MyDrive/sparta/tp4/review_analysis/data/step7_2_aspect_pairs.parquet
  총 1,576,839페어
  메모리: 507.8 MB

저장: /content/drive/MyDrive/sparta/tp4/review_analysis/data/step7_2_unmatched.parquet
  총 571,584문장

저장: /content/drive/MyDrive/sparta/tp4/review_analysis/data/step7_2_match_stats.csv


,category,matched_sentences,pct_of_total,top_keywords
4,detail,323082,21.09,"이쁘(123303), 예쁘(85043), 디자인(26882)"
1,size,320608,20.93,"사이즈(101399), 크(73691), 맞(37306)"
2,fabric,226324,14.78,"재질(39485), 두껍(28585), 기모(28213)"
0,fit,169897,11.09,"핏(103028), 오버핏(48963), 박시(5516)"
5,season,143904,9.40,"여름(31592), 따뜻(26885), 겨울(17231)"
3,color,123617,8.07,"색감(46406), 색상(26781), 블랙(6346)"
7,price,104214,6.80,"가격(49307), 가성비(34081), 싸(11871)"
6,comfort,86390,5.64,"편하(65463), 가볍(9933), 편안(3723)"
8,delivery,55871,3.65,"배송(31475), 빠르(19443), 늦(2651)"
9,durability,22932,1.50,"세탁(9431), 늘어나(4801), 튼튼(3373)"
